# Narrative Trails sobre bpoil

Línea base del proyecto: el método original de Narrative Trails sobre el subset
bpoil completo (1133 documentos, abril de 2010 a enero de 2011), sin el
truncamiento que se usa con RollingLDA. Hace lo mismo que
`scripts/baseline_narrative.py`, pero deja el landscape en memoria para probar
distintos pares de documentos sin reajustarlo.

El método tiene tres pasos:

1. UMAP proyecta los embeddings (mpnet, 768 dimensiones) a 48 dimensiones y
   HDBSCAN agrupa esa proyección en tópicos. Cada documento queda con una
   distribución de pertenencia a los tópicos.
2. La coherencia entre dos documentos es la media geométrica entre la similitud
   angular de sus embeddings y la similitud de sus distribuciones de tópicos
   (Jensen-Shannon). Se descartan las aristas más débiles que la arista más
   débil del árbol de expansión máxima (la coherencia crítica), lo que deja el
   grafo conexo. Con la restricción de fechas, cada documento solo apunta a
   documentos del mismo día o posteriores.
3. Una narrativa entre un origen y un destino es el camino de capacidad máxima:
   el que maximiza la coherencia de su eslabón más débil (el bottleneck).
   `reliability` es la media geométrica de las coherencias del camino.

Paper y repositorio en las [referencias del README](../README.md#narrative-trails).

In [1]:
import warnings

# Dos avisos que no afectan el resultado: tqdm pide ipywidgets al importarse
# desde umap, y hdbscan 0.8.40 llama a scikit-learn 1.6 con un argumento que
# se renombró ("force_all_finite").
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message=".*force_all_finite.*")

import pandas as pd

from causal_coherence.data_loading import documents_between, load_bpoil_full
from causal_coherence.narrative import (
    alternatives_summary,
    count_topics,
    extract_alternatives,
    load_or_build_landscape,
    print_alternatives,
)

pd.set_option("display.max_colwidth", None)

REFERENCE_HASH = "63d0faab065a6f4c"  # results/corrida_1.txt y corrida_2.txt

df, embeddings = load_bpoil_full()
landscape, coherence_hash = load_or_build_landscape(embeddings, df["date"])

print(f"{len(df)} documentos, de {df['date'].min().date()} a {df['date'].max().date()}")
print(f"tópicos (HDBSCAN): {count_topics(landscape.cluster_labels)}")
print(f"hash de la matriz de coherencia: {coherence_hash} (igual a la referencia: {coherence_hash == REFERENCE_HASH})")

Landscape cargado desde caché: landscape_c6fc79eb47995ef7.pkl
1133 documentos, de 2010-04-01 a 2011-01-30
tópicos (HDBSCAN): 54
hash de la matriz de coherencia: 63d0faab065a6f4c (igual a la referencia: True)


## Caché del landscape

UMAP, HDBSCAN y el grafo de coherencia tardan unos 20 segundos en este corpus.
`load_or_build_landscape` guarda el landscape ajustado en `data/cache/` (git lo
ignora) y en las corridas siguientes lo carga en menos de un segundo. El nombre
del archivo es un hash de todo lo que determina el resultado: embeddings,
fechas, `LANDSCAPE_PARAMS` y las versiones de las librerías que lo cambian (ver
`docs/reproducibilidad.md`). Si cambia cualquiera de esas cosas, cambia el hash
y el landscape se vuelve a ajustar.

Junto al landscape se guarda el hash de la matriz de coherencia, el mismo
identificador con que `scripts/baseline_narrative.py` nombra sus resultados.

## Los títulos de bpoil

En este corpus ningún documento trae titular. El campo `title` son las primeras
palabras del texto con "…" al final, y el `metadata` de cada documento lo dice
("title derived from lead"). Por eso, en cada narrativa se imprime el título
completo tal como viene y debajo los primeros 240 caracteres del contenido.

In [2]:
derived = df["metadata"].astype(str).str.contains("title derived from lead")
print(f"documentos con título derivado del texto: {derived.sum()} de {len(df)}")

documentos con título derivado del texto: 1133 de 1133


## Narrativas para el par documentado

Origen 3 (22 de abril de 2010): reporta el hundimiento de la plataforma y el
riesgo de derrame, que es el inicio real de la crisis. Destino 975 (16 de
septiembre de 2010): el pozo va a quedar sellado ese fin de semana, que es el
cierre de la historia principal. Son los extremos que justifica el comentario
de `scripts/baseline_narrative.py` y los de `results/corrida_1.txt`. El script,
en cambio, tiene fijado hoy el par 15 → 572, que se prueba en la sección
siguiente.

Las alternativas se extraen como en el script: después de cada narrativa se
ocultan sus nodos intermedios y se vuelve a buscar, hasta tener `N_PATHS`
caminos distintos o quedarse sin caminos.

In [3]:
SRC_NODE, TGT_NODE = 3, 975
N_PATHS = 3

storylines = extract_alternatives(landscape, SRC_NODE, TGT_NODE, N_PATHS)
display(alternatives_summary(storylines).round(3))
print_alternatives(df, landscape, storylines, lead_chars=240)

,largo,bottleneck,reliability
alternativa,,,
0,4,0.523,0.716
1,3,0.515,0.644
2,3,0.503,0.636


--- Alternativa 0: 4 documentos · bottleneck 0.523 · reliability 0.716
[3] 2010-04-22 · tópico 41
    Rig sinks in Gulf of Mexico, oil spill risk looms Fire boat…
    > Rig sinks in Gulf of Mexico, oil spill risk looms Fire boat response crews battle the blazing remnants of the off shore oil rig Deepwater Horizon, off Louisiana, in this handout photograph taken on April 21, 2010 and obtained on April 22. E…
[49] 2010-05-01 · tópico 41 · coherencia 0.890
    Federal and state officials pushed oil giant BP to intensify its efforts…
    > Federal and state officials pushed oil giant BP to intensify its efforts to cap a leaking oil well in the Gulf of Mexico and to contain the slick that is threatening the shores and livelihoods of people in five states. As crude oil began to…
[324] 2010-06-01 · tópico 41 · coherencia 0.523
    Tue Jun 1, 2010 1:33 pm EDT ( Reuters ) - Millions…
    > Tue Jun 1, 2010 1:33 pm EDT ( Reuters ) - Millions of gallons ( liters ) of oil have poured into the Gulf 

Con el par documentado, las tres alternativas son cortas (3 y 4 documentos) y su
bottleneck ronda 0,5, bajo comparado con el del par de la sección siguiente
(alrededor de 0,82). Los caminos saltan casi directo de fines de abril a
septiembre. El eslabón más débil siempre está en el tramo de abril a junio; la
llegada a 975 tiene coherencias de 0,79 a 0,81.

La alternativa 2 pasa por el documento 8, cuyo título es solo "Apr." porque el
texto empieza con "Apr. 21: …". Es un ejemplo del problema de títulos descrito
arriba.

## Probar otro par

Para probar otros extremos basta con cambiar `SRC_NODE` y `TGT_NODE` y correr
solo la celda: usa el landscape ya cargado. Por la restricción de fechas, el
origen tiene que ser del mismo día o anterior al destino. `documents_between`
lista candidatos en una ventana de fechas; los índices son los de `df`, que
está ordenado por fecha.

In [4]:
documents_between(df, "2010-04-26", "2010-04-28")

,date,title
9,2010-04-26,"The slick has now grown to about 1,500 sq km There are…"
10,2010-04-26,Robot vessels used to cap Gulf of Mexico oil leak The US…
11,2010-04-26,Debris and oil from the Deepwater Horizon drilling platform float in the…
12,2010-04-26,Industry officials acknowledge it could take months to entirely contain leak from…
13,2010-04-27,April 26: Weathered oil is seen on the surface of the Gulf…
14,2010-04-27,Energy firm beats expectations with # 3.6 bn quarterly profit but performance…
15,2010-04-27,"LONDON | Tue Apr 27, 2010 4:08 am EDT LONDON ( Reuters…"
16,2010-04-27,Gulf of Mexico oil spill creates environmental and political dilemmas View how…
17,2010-04-28,April 27: Weathered oil from a leaking pipeline that resulted the explosion…
18,2010-04-28,ON THE GULF OF MEXICO -- The deadly blowout of an oil…


In [5]:
SRC_NODE, TGT_NODE = 15, 572  # par fijado hoy en scripts/baseline_narrative.py
N_PATHS = 3

assert df.loc[SRC_NODE, "date"] <= df.loc[TGT_NODE, "date"], "el origen tiene que ser anterior al destino"
storylines = extract_alternatives(landscape, SRC_NODE, TGT_NODE, N_PATHS)
display(alternatives_summary(storylines).round(3))
print_alternatives(df, landscape, storylines, lead_chars=160)

,largo,bottleneck,reliability
alternativa,,,
0,7,0.828,0.849
1,8,0.816,0.848
2,6,0.809,0.827


--- Alternativa 0: 7 documentos · bottleneck 0.828 · reliability 0.849
[15] 2010-04-27 · tópico 39
    LONDON | Tue Apr 27, 2010 4:08 am EDT LONDON ( Reuters…
    > LONDON | Tue Apr 27, 2010 4:08 am EDT LONDON ( Reuters ) - BP Plc ( BP. L ) failed to reassure investors with a more than doubling of first-quarter net profits…
[28] 2010-04-29 · tópico 39 · coherencia 0.876
    LONDON | Thu Apr 29, 2010 11:29 am EDT LONDON ( Reuters…
    > LONDON | Thu Apr 29, 2010 11:29 am EDT LONDON ( Reuters ) - Shares in London-based BP Plc fell 7 percent on Thursday after the oil major said a leaking well in…
[332] 2010-06-02 · tópico 39 · coherencia 0.828
    US attorney general, Eric Holder, confirmed that a criminal and civil investigation…
    > US attorney general, Eric Holder, confirmed that a criminal and civil investigation had been opened Workers in Louisiana tackle oil from the Deepwater Horizon l…
[368] 2010-06-04 · tópico 39 · coherencia 0.834
    LONDON -- As BP shares take a pounding and

El par 15 → 572 da caminos más largos (6 a 8 documentos) y con bottleneck más
alto (0,81 a 0,83). Los extremos están separados por 54 días. Los tres caminos
empiezan con varios documentos del tópico 39 (acciones, dividendos y demandas
contra BP) y terminan en la crítica a Tony Hayward. Es una narrativa más
coherente que la del par documentado, pero sigue un hilo más estrecho de la
historia.